# 🌐 Vanilla GCN — Complete Pipeline (Google Colab)

**Two implementations of the same Vanilla GCN architecture:**
- 🔴 **Version A** — PyTorch (uses `torch.nn`, autograd)
- 🔵 **Version B** — Pure NumPy (manual forward + backprop, zero PyTorch)

**Core equation (identical in both versions):**
$$H^{(k)} = \sigma\!\left(\tilde{A}\, H^{(k-1)}\, W^{(k)}\right)$$
$$\tilde{A} = \hat{D}^{-1/2}\,\hat{A}\,\hat{D}^{-1/2}, \quad \hat{A} = A + I$$

---

## How to use your own dataset

Go to **Section 2** and follow the instructions.
You need to provide:
- **`A`** — adjacency matrix `(N, N)` as a NumPy array (binary, symmetric for undirected)
- **`X`** — node feature matrix `(N, F)` as a NumPy array
- **`y`** — integer labels `(N,)` in range `[0, C-1]`

**Supported input formats:** CSV, NPZ, NumPy `.npy`, or inline arrays.

## Section 0 — Install & Import

In [ ]:
# ── Install (only installs if not already present) ──────────────────────────
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

for pkg, import_name in [('numpy','numpy'), ('matplotlib','matplotlib'),
                          ('networkx','networkx'), ('scikit-learn','sklearn'),
                          ('torch','torch'), ('pandas','pandas')]:
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f'Installing {pkg}...')
        _pip(pkg)

print('✅ All dependencies available.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = ['#E63946', '#2A9D8F', '#F4A261', '#457B9D', '#A8DADC', '#6A4C93']

print('✅ Imports OK. PyTorch version:', torch.__version__)

---
## Section 1 — Shared: Graph Preprocessing

These functions are shared by both implementations.
$$\hat{A} = A + I, \quad \tilde{A} = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}$$

In [ ]:
# ─── Shared preprocessing (used by both PyTorch and NumPy versions) ─────────

def add_self_loops_np(A: np.ndarray) -> np.ndarray:
    """Â = A + I"""
    return A + np.eye(A.shape[0], dtype=np.float64)

def symmetric_normalize_np(A_hat: np.ndarray) -> np.ndarray:
    """Ã = D̂^(-1/2) Â D̂^(-1/2)"""
    deg = A_hat.sum(axis=1)                              # (N,)
    inv_sqrt = np.where(deg > 0, 1.0 / np.sqrt(deg), 0.0)
    D_inv_sqrt = np.diag(inv_sqrt)
    return D_inv_sqrt @ A_hat @ D_inv_sqrt

def prepare_graph_np(A, X, y):
    """Full preprocessing → (A_hat, A_tilde, X, y) as float64 NumPy arrays."""
    A = np.array(A, dtype=np.float64)
    X = np.array(X, dtype=np.float64)
    y = np.array(y, dtype=np.int64)
    A_hat   = add_self_loops_np(A)
    A_tilde = symmetric_normalize_np(A_hat)
    return A_hat, A_tilde, X, y

def prepare_graph_pt(A, X, y):
    """Full preprocessing → (A_hat, A_tilde, X, y) as float32 PyTorch tensors."""
    A_hat_np, A_tilde_np, X_np, y_np = prepare_graph_np(A, X, y)
    A_hat   = torch.tensor(A_hat_np,   dtype=torch.float32)
    A_tilde = torch.tensor(A_tilde_np, dtype=torch.float32)
    X_pt    = torch.tensor(X_np,       dtype=torch.float32)
    y_pt    = torch.tensor(y_np,       dtype=torch.long)
    return A_hat, A_tilde, X_pt, y_pt

def make_masks(N, y, train_ratio=0.6, val_ratio=0.2, seed=42):
    """Create stratified train/val/test masks."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(N)
    n_train = max(1, int(N * train_ratio))
    n_val   = max(1, int(N * val_ratio))
    train_mask = np.zeros(N, dtype=bool); train_mask[idx[:n_train]] = True
    val_mask   = np.zeros(N, dtype=bool); val_mask[idx[n_train:n_train+n_val]] = True
    test_mask  = np.zeros(N, dtype=bool); test_mask[idx[n_train+n_val:]] = True
    return train_mask, val_mask, test_mask

print('✅ Preprocessing functions defined.')

---
## Section 2 — 📊 Load YOUR Dataset

**Choose ONE of the options A–E below. Run only that cell.**

After running your chosen cell, all following sections work automatically.

At the end of each option, you will have:
```python
A            # (N, N) numpy adjacency  — binary, symmetric
X            # (N, F) numpy features
y            # (N,)   numpy int labels [0 .. C-1]
train_mask   # (N,)   bool
val_mask     # (N,)   bool
test_mask    # (N,)   bool
```

### Option A — Upload files from your computer (Colab)

In [ ]:
# ═══ OPTION A — Upload files from your computer ══════════════════════════════
# Upload 3 files:
#   adjacency.npy / adjacency.csv  — (N, N) matrix
#   features.npy  / features.csv   — (N, F) matrix
#   labels.npy    / labels.csv     — (N,)   vector
#
# You can also upload a single .npz containing all three arrays.
# Run this cell ONLY if running on Google Colab.
# ─────────────────────────────────────────────────────────────────────────────

try:
    from google.colab import files
    print('Upload your dataset files (adjacency, features, labels):')
    uploaded = files.upload()   # ← Choose your files here

    def _load(name):
        if name + '.npy' in uploaded:
            return np.load(name + '.npy', allow_pickle=True)
        elif name + '.csv' in uploaded:
            return np.loadtxt(name + '.csv', delimiter=',')
        elif name + '.npz' in uploaded:   # single archive
            d = np.load(name + '.npz')
            return d
        else:
            raise FileNotFoundError(f'{name}.npy / {name}.csv not found in upload.')

    # Check for single NPZ first
    npz_files = [k for k in uploaded if k.endswith('.npz')]
    if npz_files:
        d  = np.load(npz_files[0])
        A  = d[list(d.keys())[0]].astype(np.float64)
        X  = d[list(d.keys())[1]].astype(np.float64)
        y  = d[list(d.keys())[2]].astype(np.int64)
    else:
        A  = _load('adjacency').astype(np.float64)
        X  = _load('features').astype(np.float64)
        y  = _load('labels').astype(np.int64)

    train_mask, val_mask, test_mask = make_masks(len(y), y, seed=SEED)
    print(f'✅ Loaded: N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')
except Exception as e:
    print(f'Skipping Option A: {e}')

### Option B — Load from Google Drive

In [ ]:
# ═══ OPTION B — Load from Google Drive ═══════════════════════════════════════
# Edit the paths below to point to your files in Drive.
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_ADJ      = '/content/drive/MyDrive/gcn_data/adjacency.npy'
DRIVE_FEATURES = '/content/drive/MyDrive/gcn_data/features.npy'
DRIVE_LABELS   = '/content/drive/MyDrive/gcn_data/labels.npy'

import os
if os.path.exists(DRIVE_ADJ):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass
    A  = np.load(DRIVE_ADJ).astype(np.float64)
    X  = np.load(DRIVE_FEATURES).astype(np.float64)
    y  = np.load(DRIVE_LABELS).astype(np.int64)
    train_mask, val_mask, test_mask = make_masks(len(y), y, seed=SEED)
    print(f'✅ Loaded from Drive: N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')
else:
    print('Skipping Option B — path does not exist. Edit DRIVE_ADJ / DRIVE_FEATURES / DRIVE_LABELS.')

### Option C — Load from CSV / NPZ on disk

In [ ]:
# ═══ OPTION C — Load from CSV or NPZ paths ══════════════════════════════════
# Edit these paths.
# ─────────────────────────────────────────────────────────────────────────────

PATH_ADJ      = 'adjacency.csv'   # or .npy
PATH_FEATURES = 'features.csv'    # or .npy
PATH_LABELS   = 'labels.csv'      # or .npy

import os

def _load_auto(path):
    if path.endswith('.npy'):
        return np.load(path)
    elif path.endswith('.npz'):
        d = np.load(path)
        return d[d.files[0]]
    else:  # CSV
        return np.loadtxt(path, delimiter=',')

if os.path.exists(PATH_ADJ):
    A  = _load_auto(PATH_ADJ).astype(np.float64)
    X  = _load_auto(PATH_FEATURES).astype(np.float64)
    y  = _load_auto(PATH_LABELS).astype(np.int64)
    train_mask, val_mask, test_mask = make_masks(len(y), y, seed=SEED)
    print(f'✅ Loaded: N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')
else:
    print('Skipping Option C — file not found. Edit PATH_ADJ / PATH_FEATURES / PATH_LABELS.')

### Option D — Paste inline NumPy arrays

In [ ]:
# ═══ OPTION D — Paste your arrays inline ════════════════════════════════════
# Replace the placeholders below with your actual data.
# ─────────────────────────────────────────────────────────────────────────────

# EXAMPLE — replace these with your own:
# A = np.array([[0,1,0],[1,0,1],[0,1,0]], dtype=np.float64)   # N=3, undirected
# X = np.array([[1,0],[0,1],[1,1]],       dtype=np.float64)   # 2 features
# y = np.array([0, 1, 0],                dtype=np.int64)      # 2 classes

# ← Uncomment and edit the lines above, then run this cell.
# train_mask, val_mask, test_mask = make_masks(len(y), y, seed=SEED)
# print(f'✅ N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')
print('Option D: uncomment the code above and edit with your data.')

### Option E — Use built-in synthetic dataset (default / fallback)

In [ ]:
# ═══ OPTION E — Synthetic 12-node 3-community graph (fallback) ══════════════
# This is always safe to run. If you ran Options A–D successfully,
# this cell will simply print the current dataset info.
# ─────────────────────────────────────────────────────────────────────────────

def _make_synthetic(seed=42):
    """12-node 3-class community graph with Gaussian features."""
    rng = np.random.default_rng(seed)
    N_per_class, n_classes, n_features = 4, 3, 4
    N = N_per_class * n_classes

    # Adjacency: dense within community, sparse between
    A = np.zeros((N, N), dtype=np.float64)
    for c in range(n_classes):
        start = c * N_per_class
        for i in range(start, start + N_per_class):
            for j in range(start, start + N_per_class):
                if i != j:
                    A[i, j] = 1.0
    # Inter-community edges
    bridges = [(0, 4), (4, 8), (1, 9), (3, 7)]
    for u, v in bridges:
        A[u, v] = A[v, u] = 1.0

    # Features: class-specific Gaussian
    means = rng.uniform(-2, 2, size=(n_classes, n_features))
    labels = np.repeat(np.arange(n_classes), N_per_class)
    X = np.vstack([rng.normal(means[c], 0.3, size=(N_per_class, n_features))
                   for c in range(n_classes)])
    return A, X, labels

try:
    _ = A
    print(f'✅ Using existing dataset: N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')
except NameError:
    A, X, y = _make_synthetic(SEED)
    train_mask, val_mask, test_mask = make_masks(len(y), y, seed=SEED)
    print(f'✅ Synthetic dataset: N={A.shape[0]}, F={X.shape[1]}, C={y.max()+1}')

# Print dataset summary
print(f'   Nodes: {A.shape[0]}')
print(f'   Edges: {int(A.sum()) // 2}')
print(f'   Features: {X.shape[1]}')
print(f'   Classes: {y.max()+1}  |  distribution: {np.bincount(y)}')
print(f'   Train/Val/Test: {train_mask.sum()}/{val_mask.sum()}/{test_mask.sum()}')

In [ ]:
# ── Visualise the graph coloured by true class ───────────────────────────────

def plot_graph(A, y, title='Graph (True Labels)', predictions=None, ax=None):
    G = nx.from_numpy_array(A)
    G.remove_edges_from(nx.selfloop_edges(G))
    pos = nx.spring_layout(G, seed=42)
    color_labels = predictions if predictions is not None else y
    node_colors = [COLORS[int(c) % len(COLORS)] for c in color_labels]
    border_colors = ['black' if predictions is None or predictions[i] == y[i]
                     else 'red' for i in range(len(y))]
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edgecolors=border_colors, linewidths=2,
                     node_size=700, font_color='white', font_weight='bold',
                     edge_color='#aaaaaa', width=1.5, alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    legend = [mpatches.Patch(color=COLORS[c], label=f'Class {c}')
              for c in range(y.max()+1)]
    ax.legend(handles=legend, loc='upper right', framealpha=0.9)

plot_graph(A, y, title='Graph — True Labels (before GCN)')
plt.tight_layout()
plt.show()

---
## Section 3 — ⚙️ Hyperparameters

Edit these before running the training sections.

In [ ]:
# ─── Hyperparameters (shared by both versions) ───────────────────────────────

HIDDEN_DIM   = 16        # hidden dimension
NUM_LAYERS   = 2         # GCN depth (1 = linear, 2 = standard, ≥3 = deep)
EPOCHS       = 200       # training epochs
LR           = 0.01      # learning rate
WEIGHT_DECAY = 5e-4      # L2 regularisation
LOG_EVERY    = 50        # print progress every N epochs

# Derived
N           = A.shape[0]
INPUT_DIM   = X.shape[1]
NUM_CLASSES = int(y.max()) + 1

print(f'Architecture: GCN({INPUT_DIM} → {HIDDEN_DIM} × {NUM_LAYERS-1} → {NUM_CLASSES})')
print(f'Training: {EPOCHS} epochs, lr={LR}, weight_decay={WEIGHT_DECAY}')

---
## Section 4 — 🔴 Version A: PyTorch GCN

Uses `torch.nn.Module` and autograd. The graph convolution is still manually written as `A_tilde @ H @ W` — no PyG/DGL.

In [ ]:
# ─── PyTorch GCN Layer ───────────────────────────────────────────────────────

class GCNLayerPT(nn.Module):
    """Single GCN layer: H_out = σ(Ã @ H @ W)."""

    def __init__(self, input_dim, output_dim, activation=F.relu):
        super().__init__()
        self.input_dim  = input_dim
        self.output_dim = output_dim
        self.activation = activation
        # Learnable weight matrix (no bias — matching the GCN paper)
        self.weight = nn.Parameter(torch.empty(input_dim, output_dim))
        nn.init.xavier_uniform_(self.weight)   # Glorot initialisation

    def forward(self, H, A_tilde):
        support = A_tilde @ H           # neighbour aggregation
        output  = support @ self.weight  # linear projection
        return self.activation(output) if self.activation else output


class VanillaGCN_PT(nn.Module):
    """K-layer Vanilla GCN (PyTorch).
    H^(k) = σ(Ã H^(k-1) W^(k))
    """

    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2):
        super().__init__()
        if num_layers < 1:
            raise ValueError('num_layers must be >= 1')
        layers = []
        if num_layers == 1:
            layers.append(GCNLayerPT(input_dim, num_classes, activation=None))
        else:
            layers.append(GCNLayerPT(input_dim, hidden_dim, activation=F.relu))
            for _ in range(num_layers - 2):
                layers.append(GCNLayerPT(hidden_dim, hidden_dim, activation=F.relu))
            layers.append(GCNLayerPT(hidden_dim, num_classes, activation=None))
        self.layers = nn.ModuleList(layers)

    def forward(self, X, A_tilde, return_intermediate=False):
        H = X
        intermediates = {'input': H}
        for k, layer in enumerate(self.layers):
            H = layer(H, A_tilde)
            key = f'layer_{k+1}' if k < len(self.layers)-1 else 'logits'
            intermediates[key] = H
        return intermediates if return_intermediate else H

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


print('✅ PyTorch GCN classes defined.')

In [ ]:
# ─── PyTorch Training ────────────────────────────────────────────────────────

import time
torch.manual_seed(SEED)

# Preprocess
_, A_tilde_pt, X_pt, y_pt = prepare_graph_pt(A, X, y)
train_mask_pt = torch.tensor(train_mask)
val_mask_pt   = torch.tensor(val_mask)
test_mask_pt  = torch.tensor(test_mask)

# Build model
model_pt = VanillaGCN_PT(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, NUM_LAYERS)
print(f'PyTorch GCN | Parameters: {model_pt.count_parameters():,}')

optimizer = optim.Adam(model_pt.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

pt_history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model_pt.train()
    optimizer.zero_grad()
    logits = model_pt(X_pt, A_tilde_pt)
    loss   = criterion(logits[train_mask_pt], y_pt[train_mask_pt])
    loss.backward()
    optimizer.step()

    model_pt.eval()
    with torch.no_grad():
        logits_v = model_pt(X_pt, A_tilde_pt)
        v_loss   = criterion(logits_v[val_mask_pt], y_pt[val_mask_pt]).item()
        def _acc(mask):
            preds = logits_v[mask].argmax(1)
            return (preds == y_pt[mask]).float().mean().item() * 100
        t_acc = _acc(train_mask_pt)
        v_acc = _acc(val_mask_pt)

    pt_history['train_loss'].append(loss.detach().item())
    pt_history['val_loss'].append(v_loss)
    pt_history['train_acc'].append(t_acc)
    pt_history['val_acc'].append(v_acc)

    if epoch % LOG_EVERY == 0:
        print(f'[PT] Ep {epoch:>4d}/{EPOCHS} | loss={loss.item():.4f} | '
              f'val_loss={v_loss:.4f} | train_acc={t_acc:.1f}% | val_acc={v_acc:.1f}%')

print(f'\n✅ PyTorch training done in {time.time()-t0:.1f}s')

# Final test accuracy
model_pt.eval()
with torch.no_grad():
    logits_final_pt = model_pt(X_pt, A_tilde_pt)
    preds_pt = logits_final_pt.argmax(1).numpy()
    test_acc_pt = (preds_pt[test_mask] == y[test_mask]).mean() * 100
    print(f'   Test accuracy (PT): {test_acc_pt:.1f}%')

---
## Section 5 — 🔵 Version B: Pure NumPy GCN

Same architecture. Zero PyTorch. All gradients computed by hand.

$$
\text{Forward:}\quad S = \tilde{A} H, \quad Z = SW, \quad H_{\text{out}} = \sigma(Z)
$$
$$
\text{Backward:}\quad \frac{\partial L}{\partial W} = S^\top \frac{\partial L}{\partial Z}, \quad
\frac{\partial L}{\partial H} = \tilde{A}^\top \left(\frac{\partial L}{\partial Z} W^\top\right)
$$

In [ ]:
# ─── NumPy GCN Layer (manual forward + backward) ─────────────────────────────

def _glorot(fan_in, fan_out, rng):
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limit, limit, size=(fan_in, fan_out))

class GCNLayerNP:
    """Single GCN layer (NumPy): H_out = σ(Ã @ H @ W)."""

    def __init__(self, input_dim, output_dim, activation='relu', seed=42):
        self.input_dim  = input_dim
        self.output_dim = output_dim
        self.activation = activation   # 'relu' or None
        rng = np.random.default_rng(seed)
        self.W  = _glorot(input_dim, output_dim, rng)   # (in, out)
        self.dW = None
        self._cache = {}

    def forward(self, H, A_tilde):
        S = A_tilde @ H             # (N, in) — aggregation
        Z = S @ self.W              # (N, out) — linear
        H_out = np.maximum(0, Z) if self.activation == 'relu' else Z
        self._cache = {'S': S, 'Z': Z, 'A_tilde': A_tilde}
        return H_out

    def backward(self, dL_dH_out):
        S, Z, At = self._cache['S'], self._cache['Z'], self._cache['A_tilde']
        dL_dZ   = dL_dH_out * (Z > 0) if self.activation == 'relu' else dL_dH_out
        self.dW = S.T @ dL_dZ              # (in, out)
        dL_dH   = At.T @ (dL_dZ @ self.W.T)  # (N, in)
        return dL_dH

    def step(self, lr, wd=0.0):
        self.W -= lr * (self.dW + wd * self.W)
        self.dW = None


class VanillaGCN_NP:
    """K-layer Vanilla GCN (pure NumPy).
    H^(k) = σ(Ã H^(k-1) W^(k))
    """

    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, seed=42):
        if num_layers < 1:
            raise ValueError('num_layers must be >= 1')
        self.layers = []
        if num_layers == 1:
            self.layers.append(GCNLayerNP(input_dim, num_classes, None, seed))
        else:
            self.layers.append(GCNLayerNP(input_dim, hidden_dim, 'relu', seed))
            for k in range(1, num_layers - 1):
                self.layers.append(GCNLayerNP(hidden_dim, hidden_dim, 'relu', seed+k))
            self.layers.append(GCNLayerNP(hidden_dim, num_classes, None, seed+num_layers))

    def forward(self, X, A_tilde, return_intermediate=False):
        H = X.copy(); outs = {'input': H.copy()}
        for k, layer in enumerate(self.layers):
            H = layer.forward(H, A_tilde)
            if return_intermediate:
                key = f'layer_{k+1}' if k < len(self.layers)-1 else 'logits'
                outs[key] = H.copy()
        return outs if return_intermediate else H

    def backward(self, dL_dlogits):
        grad = dL_dlogits
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def step(self, lr, wd=0.0):
        for layer in self.layers:
            layer.step(lr, wd)

    def count_parameters(self):
        return sum(layer.W.size for layer in self.layers)


print('✅ NumPy GCN classes defined.')

In [ ]:
# ─── NumPy Training ──────────────────────────────────────────────────────────

np.random.seed(SEED)

# Preprocess
_, A_tilde_np, X_np, y_np = prepare_graph_np(A, X, y)

# Loss helpers
def softmax_np(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_np(probs, y, mask):
    n = mask.sum()
    return float(-np.log(np.clip(probs[mask, y[mask]], 1e-12, 1)).mean()) if n else 0.0

def ce_backward_np(probs, y, mask):
    dL = np.zeros_like(probs)
    n  = mask.sum()
    if n > 0:
        dL[mask] = (probs[mask] - np.eye(probs.shape[1])[y[mask]]) / n
    return dL

def acc_np(probs, y, mask):
    n = mask.sum()
    return float((probs[mask].argmax(1) == y[mask]).mean()*100) if n else 0.0

# Build model
model_np = VanillaGCN_NP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, NUM_LAYERS, SEED)
print(f'NumPy GCN | Parameters: {model_np.count_parameters():,}')

np_history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    # Forward
    logits = model_np.forward(X_np, A_tilde_np)
    probs  = softmax_np(logits)

    t_loss = cross_entropy_np(probs, y_np, train_mask)
    v_loss = cross_entropy_np(probs, y_np, val_mask)
    t_acc  = acc_np(probs, y_np, train_mask)
    v_acc  = acc_np(probs, y_np, val_mask)

    # Backward + update
    dL = ce_backward_np(probs, y_np, train_mask)
    model_np.backward(dL)
    model_np.step(LR, WEIGHT_DECAY)

    np_history['train_loss'].append(t_loss)
    np_history['val_loss'].append(v_loss)
    np_history['train_acc'].append(t_acc)
    np_history['val_acc'].append(v_acc)

    if epoch % LOG_EVERY == 0:
        print(f'[NP] Ep {epoch:>4d}/{EPOCHS} | loss={t_loss:.4f} | '
              f'val_loss={v_loss:.4f} | train_acc={t_acc:.1f}% | val_acc={v_acc:.1f}%')

print(f'\n✅ NumPy training done in {time.time()-t0:.1f}s')

# Final test accuracy
logits_np = model_np.forward(X_np, A_tilde_np)
preds_np  = logits_np.argmax(axis=1)
test_acc_np = (preds_np[test_mask] == y_np[test_mask]).mean() * 100
print(f'   Test accuracy (NP): {test_acc_np:.1f}%')

---
## Section 6 — 📈 Compare & Visualise

In [ ]:
# ─── Training curves side by side ────────────────────────────────────────────

epochs_x = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Training Curves: 🔴 PyTorch vs 🔵 NumPy', fontsize=14, fontweight='bold')

pairs = [
    ('train_loss', 'Training Loss'),
    ('val_loss',   'Validation Loss'),
    ('train_acc',  'Training Accuracy (%)'),
    ('val_acc',    'Validation Accuracy (%)'),
]
for ax, (key, title) in zip(axes.flat, pairs):
    ax.plot(epochs_x, pt_history[key], color='#E63946', lw=2, label='PyTorch')
    ax.plot(epochs_x, np_history[key], color='#2A9D8F', lw=2, label='NumPy', linestyle='--')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: training_curves.png')

In [ ]:
# ─── Graph: Before vs After (PyTorch vs NumPy) ───────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Node Classification Results', fontsize=14, fontweight='bold')

plot_graph(A, y, 'True Labels\n(ground truth)', ax=axes[0])
plot_graph(A, y, f'🔴 PyTorch GCN Predictions\nTest acc: {test_acc_pt:.1f}%',
           predictions=preds_pt, ax=axes[1])
plot_graph(A, y, f'🔵 NumPy GCN Predictions\nTest acc: {test_acc_np:.1f}%',
           predictions=preds_np, ax=axes[2])

plt.tight_layout()
plt.savefig('predictions.png', bbox_inches='tight', dpi=150)
plt.show()
print('(Red borders = misclassified nodes)')

In [ ]:
# ─── PCA of final node embeddings ────────────────────────────────────────────

def pca2d(Z):
    if isinstance(Z, torch.Tensor):
        Z = Z.detach().numpy()
    if Z.shape[1] == 1:
        return np.column_stack([Z, np.zeros(len(Z))])
    if Z.shape[1] == 2:
        return Z
    pca = PCA(n_components=2, random_state=42)
    return pca.fit_transform(Z)

# Get embeddings
model_pt.eval()
with torch.no_grad():
    embs_pt = model_pt(X_pt, A_tilde_pt, return_intermediate=True)
    # Use last hidden layer (penultimate)
    hidden_keys_pt = [k for k in embs_pt if k not in ('input', 'logits')]
    Z_pt = embs_pt[hidden_keys_pt[-1]] if hidden_keys_pt else embs_pt['input']

embs_np = model_np.forward(X_np, A_tilde_np, return_intermediate=True)
hidden_keys_np = [k for k in embs_np if k not in ('input', 'logits')]
Z_np = embs_np[hidden_keys_np[-1]] if hidden_keys_np else embs_np['input']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Node Embeddings (PCA)', fontsize=14, fontweight='bold')

for ax, (Z, title) in zip(axes, [
    (X_np,                    'Input features X'),
    (Z_pt.numpy() if isinstance(Z_pt, torch.Tensor) else Z_pt, '🔴 PyTorch last hidden'),
    (Z_np,                    '🔵 NumPy last hidden'),
]):
    coords = pca2d(Z)
    for c in range(NUM_CLASSES):
        mask_c = y == c
        ax.scatter(coords[mask_c, 0], coords[mask_c, 1],
                   c=COLORS[c % len(COLORS)], s=120, label=f'Class {c}',
                   edgecolors='white', linewidths=0.8, zorder=3)
    for i in range(N):
        ax.annotate(str(i), coords[i], fontsize=7, ha='center', va='center',
                    fontweight='bold', color='white')
    ax.set_title(title, fontsize=11); ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2); ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')

plt.tight_layout()
plt.savefig('embeddings_pca.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ─── Summary table ───────────────────────────────────────────────────────────

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
print(f'{"":30s} {"PyTorch":>12} {"NumPy":>10}')
print('-' * 60)
print(f'{"Architecture":30s} {f"GCN-{NUM_LAYERS}L":>12} {f"GCN-{NUM_LAYERS}L":>10}')
print(f'{"Parameters":30s} {model_pt.count_parameters():>12,} {model_np.count_parameters():>10,}')
print(f'{"Epochs":30s} {EPOCHS:>12} {EPOCHS:>10}')
print(f'{"Final train loss":30s} {pt_history["train_loss"][-1]:>12.4f} {np_history["train_loss"][-1]:>10.4f}')
print(f'{"Final val loss":30s} {pt_history["val_loss"][-1]:>12.4f} {np_history["val_loss"][-1]:>10.4f}')
print(f'{"Best val acc":30s} {max(pt_history["val_acc"]):>11.1f}% {max(np_history["val_acc"]):>9.1f}%')
print(f'{"Test acc":30s} {test_acc_pt:>11.1f}% {test_acc_np:>9.1f}%')
print('=' * 60)
print('Both use: Ã = D̂^(-1/2)(A+I)D̂^(-1/2), H^(k) = σ(Ã H^(k-1) W^(k))')
print('PyTorch: uses autograd. NumPy: manual forward + backward.')

---
## Section 7 — 🔍 Over-Smoothing Analysis

As GCN depth increases, all node embeddings converge → classification degrades.

In [ ]:
# ─── Over-smoothing depth sweep ──────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

def mean_pairwise_cosine_sim(Z):
    if isinstance(Z, torch.Tensor):
        Z = Z.detach().numpy()
    sim = sk_cosine(Z)  # (N, N)
    N = sim.shape[0]
    mask = ~np.eye(N, dtype=bool)
    return float(sim[mask].mean())

DEPTHS       = [1, 2, 3, 5, 8, 10]
SWEEP_EPOCHS = EPOCHS

pt_sims, pt_accs = [], []
np_sims, np_accs = [], []

print('Depth sweep...')
for depth in DEPTHS:
    # ── PyTorch ──
    torch.manual_seed(SEED)
    m_pt = VanillaGCN_PT(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, depth)
    opt  = optim.Adam(m_pt.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    for _ in range(SWEEP_EPOCHS):
        m_pt.train(); opt.zero_grad()
        l = nn.CrossEntropyLoss()(m_pt(X_pt, A_tilde_pt)[train_mask_pt], y_pt[train_mask_pt])
        l.backward(); opt.step()
    m_pt.eval()
    with torch.no_grad():
        embs_sweep = m_pt(X_pt, A_tilde_pt, return_intermediate=True)
        hk = [k for k in embs_sweep if k not in ('input','logits')]
        Z_s = embs_sweep[hk[-1]] if hk else embs_sweep['input']
        preds_s = m_pt(X_pt, A_tilde_pt).argmax(1).numpy()
    pt_sims.append(mean_pairwise_cosine_sim(Z_s))
    pt_accs.append((preds_s[test_mask] == y[test_mask]).mean() * 100)

    # ── NumPy ──
    np.random.seed(SEED)
    m_np = VanillaGCN_NP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, depth, SEED)
    for _ in range(SWEEP_EPOCHS):
        lg = m_np.forward(X_np, A_tilde_np)
        pr = softmax_np(lg)
        m_np.backward(ce_backward_np(pr, y_np, train_mask))
        m_np.step(LR, WEIGHT_DECAY)
    lg_f = m_np.forward(X_np, A_tilde_np)
    preds_n = lg_f.argmax(1)
    hk_np = [k for k in m_np.forward(X_np, A_tilde_np, return_intermediate=True) if k not in ('input','logits')]
    Z_np_s_d = m_np.forward(X_np, A_tilde_np, return_intermediate=True)
    Z_np_s   = Z_np_s_d[hk_np[-1]] if hk_np else Z_np_s_d['input']
    np_sims.append(mean_pairwise_cosine_sim(Z_np_s))
    np_accs.append((preds_n[test_mask] == y_np[test_mask]).mean() * 100)

    print(f'  K={depth:2d} | PT sim={pt_sims[-1]:.3f} acc={pt_accs[-1]:.1f}% | '
          f'NP sim={np_sims[-1]:.3f} acc={np_accs[-1]:.1f}%')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Over-Smoothing: Mean Pairwise Cosine Similarity vs Depth', fontsize=13, fontweight='bold')

ax1.plot(DEPTHS, pt_sims, 'o-', color='#E63946', lw=2, label='PyTorch')
ax1.plot(DEPTHS, np_sims, 's--', color='#2A9D8F', lw=2, label='NumPy')
ax1.set_xlabel('GCN Depth (K)'); ax1.set_ylabel('Mean Cosine Similarity')
ax1.set_title('Similarity (higher = more over-smoothed)'); ax1.set_ylim(0, 1.05)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(DEPTHS, pt_accs, 'o-', color='#E63946', lw=2, label='PyTorch')
ax2.plot(DEPTHS, np_accs, 's--', color='#2A9D8F', lw=2, label='NumPy')
ax2.set_xlabel('GCN Depth (K)'); ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Test Accuracy vs Depth'); ax2.set_ylim(0, 105)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('oversmoothing.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Section 8 — 🔵 Receptive Field

A K-layer GCN aggregates K-hop neighborhood information.

In [ ]:
# ─── K-hop receptive field ───────────────────────────────────────────────────

def k_hop_neighbors(A_bin, node, max_k=3):
    """Return dict k → set of nodes reachable in exactly k hops."""
    visited = {node}
    frontier = {node}
    result = {}
    for k in range(1, max_k + 1):
        next_frontier = set()
        for v in frontier:
            for u in np.where(A_bin[v] > 0)[0]:
                if u not in visited:
                    next_frontier.add(int(u))
        visited |= next_frontier
        result[k] = next_frontier
        frontier = next_frontier
        if not frontier:
            break
    return result

TARGET = 0   # ← change this to any node index you like
HOP_COLORS_RF = ['#CCCCCC', '#2A9D8F', '#F4A261', '#457B9D']

hop_sets = k_hop_neighbors(A, TARGET, max_k=NUM_LAYERS)
print(f'Node {TARGET} receptive field (max_k={NUM_LAYERS}):')
for k, s in hop_sets.items():
    print(f'  {k}-hop: {sorted(s)}')

G = nx.from_numpy_array(A); G.remove_edges_from(nx.selfloop_edges(G))
pos = nx.spring_layout(G, seed=42)
colors = []
for i in range(N):
    if i == TARGET:
        colors.append('#E63946')
    else:
        assigned = False
        for k, s in hop_sets.items():
            if i in s:
                colors.append(HOP_COLORS_RF[k % len(HOP_COLORS_RF)])
                assigned = True; break
        if not assigned:
            colors.append(HOP_COLORS_RF[0])

fig, ax = plt.subplots(figsize=(8, 5))
nx.draw_networkx(G, pos, ax=ax, node_color=colors, node_size=700,
                 font_color='white', font_weight='bold',
                 edge_color='#aaaaaa', width=1.5)
legend = ([mpatches.Patch(color='#E63946', label=f'Target (node {TARGET})'),]
          + [mpatches.Patch(color=HOP_COLORS_RF[k], label=f'{k}-hop ({len(hop_sets.get(k,set()))} nodes)')
             for k in range(1, NUM_LAYERS+1)]
          + [mpatches.Patch(color=HOP_COLORS_RF[0], label='Outside')])
ax.legend(handles=legend, loc='upper right', framealpha=0.9)
ax.set_title(f'Receptive Field of Node {TARGET} (K={NUM_LAYERS} layers)', fontsize=12, fontweight='bold')
ax.axis('off'); plt.tight_layout()
plt.savefig('receptive_field.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Section 9 — 💾 Save Outputs

In [ ]:
# ─── Save embeddings and model weights ────────────────────────────────────────

# PyTorch checkpoint
model_pt.eval()
torch.save({
    'model_state_dict': model_pt.state_dict(),
    'config': {'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM,
               'num_classes': NUM_CLASSES, 'num_layers': NUM_LAYERS},
    'test_acc_pt': test_acc_pt,
}, 'gcn_pytorch_checkpoint.pt')
print('✅ PyTorch checkpoint: gcn_pytorch_checkpoint.pt')

# NumPy weights
np.savez('gcn_numpy_weights.npz',
         **{f'W_{i}': layer.W for i, layer in enumerate(model_np.layers)})
print('✅ NumPy weights: gcn_numpy_weights.npz')

# Final embeddings
with torch.no_grad():
    final_embs_pt = model_pt(X_pt, A_tilde_pt, return_intermediate=True)
    hkeys = [k for k in final_embs_pt if k not in ('input','logits')]
    Z_final = final_embs_pt[hkeys[-1]].numpy() if hkeys else X_pt.numpy()
np.save('node_embeddings_pt.npy', Z_final)
print('✅ Node embeddings (PT): node_embeddings_pt.npy')

# Download from Colab
try:
    from google.colab import files
    for fname in ['training_curves.png', 'predictions.png', 'embeddings_pca.png',
                  'oversmoothing.png', 'gcn_pytorch_checkpoint.pt', 'node_embeddings_pt.npy']:
        import os
        if os.path.exists(fname):
            files.download(fname)
    print('✅ Files downloaded to your machine.')
except ImportError:
    print('(Not on Colab — files saved in current directory.)')

---
## Appendix — Mathematical Reference

| Step | Formula | Code |
|------|---------|------|
| Self-loops | $\hat{A} = A + I$ | `A + np.eye(N)` |
| Degree | $\hat{D}_{ii} = \sum_j \hat{A}_{ij}$ | `A_hat.sum(axis=1)` |
| Normalize | $\tilde{A} = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}$ | `D_inv_sqrt @ A_hat @ D_inv_sqrt` |
| GCN fwd | $H^{(k)} = \sigma(\tilde{A}\,H^{(k-1)}\,W^{(k)})$ | `relu(A_tilde @ H @ W)` |
| GCN bwd W | $\partial L/\partial W = S^\top (\partial L/\partial Z)$ | `S.T @ dL_dZ` |
| GCN bwd H | $\partial L/\partial H = \tilde{A}^\top(\partial L/\partial Z\,W^\top)$ | `A_tilde.T @ (dL_dZ @ W.T)` |
| CE loss | $L = -\frac{1}{n_{\text{train}}}\sum_{i\in\text{train}}\log p_{iy_i}$ | `cross_entropy(probs, y, mask)` |
| CE bwd | $\partial L/\partial z_i = p_i - e_{y_i}$ (train only) | `probs - one_hot(y)` |

**Reference:** Hamilton (2020) *Graph Representation Learning*, Ch. 5 & 7.